# Tuesday · Lab 2 — Go Convolutional

In Lab 1 your network **flattened** each digit into a row of 784 numbers. That works — but it throws away *where* the pixels are. Today you'll build a **convolutional neural network (CNN)** that keeps the image's 2D shape and slides small **filters** across it to find edges, curves, and strokes.

This is the same idea behind **MobileNet**, the network you'll reuse tomorrow. Building a tiny one yourself makes tomorrow's transfer learning click.

**Goal:** fill in the `TODO`s, train, and break **99%** test accuracy.

You only need to complete the lines marked `# TODO`. Open the **Hint** under each one if you get stuck.

> **How to run this:** You're on our GPU server through **JupyterHub**, in your browser — everything is installed. Run each cell with **Shift+Enter**, top to bottom. When the setup cell prints `Training on: cuda`, you're on the GPU. Complete a `TODO` cell, then run it again.

In [ ]:
# Setup — given, just run it. (torch, torchvision, matplotlib are already installed here.)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Training on:", DEVICE)

## 1. The data: MNIST (given)

Same 70,000 handwritten digits as Lab 1 — but notice we do **not** flatten them. Each batch stays shaped `(batch, 1, 28, 28)`: 1 grayscale channel, 28×28 pixels. The CNN wants that 2D shape.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_ds = datasets.MNIST("data", train=True,  download=True, transform=transform)
test_ds  = datasets.MNIST("data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=64,   shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=1000, shuffle=False)

imgs, labels = next(iter(train_loader))
print("one batch of images has shape:", tuple(imgs.shape))   # (64, 1, 28, 28) — still 2D!

## 2. The building blocks

A CNN stacks a few simple layers. You'll use four:

| Layer | What it does |
|---|---|
| `nn.Conv2d(in, out, kernel_size=3, padding=1)` | Slides `out` small filters over the image, each finding one pattern. `padding=1` keeps height/width the same. |
| `F.relu(x)` | The non-linear bend — negatives become 0. |
| `nn.MaxPool2d(2)` | Zooms out: keeps the strongest value in each 2×2 block, **halving** height and width. |
| `nn.Linear(in, out)` | The plain fully-connected layer from Lab 1, used at the very end to produce 10 scores. |

The plan: **conv → pool → conv → pool → flatten → linear → linear**. Two pools turn 28×28 into 7×7.

## 3. Build the network — your turn

Fill in the three `TODO`s below. Keep `kernel_size=3, padding=1` on the convs.

In [ ]:
class DigitCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO 1: choose how many filters each conv layer learns (out_channels).
        #         A good first try: 32 for conv1, then 64 for conv2.
        #         conv2's in_channels must equal conv1's out_channels.
        self.conv1 = nn.Conv2d(1,    ____, kernel_size=3, padding=1)   # input is 1 (grayscale)
        self.conv2 = nn.Conv2d(____, ____, kernel_size=3, padding=1)
        self.pool  = nn.MaxPool2d(2)                                   # halves H and W

        # TODO 2: after two pools, 28x28 -> 14x14 -> 7x7. With 64 filters, one image
        #         becomes 64 * 7 * 7 numbers. Put that flattened size here.
        self.fc1 = nn.Linear(____, 128)
        self.fc2 = nn.Linear(128, 10)                                 # 10 digit classes

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))   # 28 -> 14
        x = self.pool(F.relu(self.conv2(x)))   # 14 -> 7
        # TODO 3: flatten x from (batch, 64, 7, 7) to (batch, 64*7*7) so Linear can use it.
        x = ____
        x = F.relu(self.fc1(x))
        return self.fc2(x)                     # raw scores (logits) for 0-9

<details><summary>Hints for TODO 1-3</summary>

- **TODO 1:** `self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)` and `self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)`.
- **TODO 2:** `64 * 7 * 7` = `3136`, so `self.fc1 = nn.Linear(64 * 7 * 7, 128)`.
- **TODO 3:** `x = x.view(x.size(0), -1)` — keep the batch dimension, flatten the rest.

</details>

## 4. Check the structure

Run this after finishing the `TODO`s. It builds the model and counts its knobs. Compare the parameter count to Lab 1's MLP (~109,000) — the CNN often does **better with a similar or smaller count**, because filters are reused across the whole image.

In [ ]:
model = DigitCNN().to(DEVICE)
print(model)
n_params = sum(p.numel() for p in model.parameters())
print(f"\nThis CNN has {n_params:,} parameters.")

# quick shape sanity check: one batch in -> (64, 10) out
out = model(imgs.to(DEVICE))
print("output shape:", tuple(out.shape), "(should be (64, 10))")

## 5. Train it (given)

Same training loop as Lab 1: show a batch, measure loss, nudge the weights. A CNN is heavier, so each epoch is slower — but it learns more from each image. Three epochs should clear 99%.

In [ ]:
def accuracy(model, loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            preds = model(imgs).argmax(1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
    return correct / total

opt     = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

EPOCHS = 3
for epoch in range(1, EPOCHS + 1):
    model.train()
    for imgs_b, labels_b in train_loader:
        imgs_b, labels_b = imgs_b.to(DEVICE), labels_b.to(DEVICE)
        opt.zero_grad()
        loss = loss_fn(model(imgs_b), labels_b)
        loss.backward()
        opt.step()
    print(f"epoch {epoch}:  test accuracy = {accuracy(model, test_loader):.4f}")

## 6. Did you break 99%?

If your final test accuracy is **0.99 or higher**, your CNN beat the Lab 1 MLP (which topped out around 97-98%). Same data, same training loop — the difference is that the CNN *looks at shape*.

If you're stuck below that, check: are the conv channels 32 then 64? Is `fc1` fed `64*7*7`? Did you run all three epochs?

## 7. Your turn — experiments

Try these and watch the test accuracy (and parameter count) move:

- **A.** In `DigitCNN`, change conv2 from 64 filters to 128. Re-run everything. Does accuracy improve? What happens to the parameter count?
- **B.** Add `nn.Dropout(0.25)` in `forward` right before `fc1` (as `x = F.dropout(x, 0.25, self.training)`). Does the gap between train and test behavior change?
- **C. Stretch:** run the cell below to *see* the filters your first conv layer learned.

In [ ]:
# Stretch: visualize the 32 filters conv1 learned (given).
import matplotlib.pyplot as plt

filters = model.conv1.weight.data.cpu()   # shape (out_channels, 1, 3, 3)
n = filters.shape[0]
cols = 8
rows = (n + cols - 1) // cols
plt.figure(figsize=(cols, rows))
for i in range(n):
    plt.subplot(rows, cols, i + 1)
    plt.imshow(filters[i, 0], cmap="gray")
    plt.axis("off")
plt.suptitle("What conv1 learned to look for")
plt.tight_layout(); plt.show()

## Wrap-up

You just built a network that sees **shape**, not just a list of numbers — and it beat yesterday's flat network on the same digits.

**Tomorrow** you won't build a CNN from scratch. You'll take **MobileNet** — a CNN already trained on millions of images — and retrain just its top layer on *your* objects. Because the hard part (learning to see edges and textures) is already done, about **50 photos per object** will be enough.